# 🇺🇸 NB8 — NOAA NWS Alerts API
**Source:** `https://api.weather.gov/alerts/active`  
**Protocol:** HTTP GET  
**Schedule:** Every 15 minutes (same pre-check gate as GDACS)  
**Auth:** None

The National Weather Service (NWS) is an agency of the United States federal government that is tasked with providing weather forecasts, warnings of hazardous weather, and other weather-related products to organizations and the public for the purposes of protection, safety, and general information. It is a part of the National Oceanic and Atmospheric Administration (NOAA) branch of the Department of Commerce. https://en.wikipedia.org/wiki/National_Weather_Service 

Fetches active Tsunami Warnings, Tsunami Watches, and Tsunami Advisories from the official US National Weather Service. Covers the US, Hawaii, Alaska, and Pacific territories. Always polled alongside GDACS for maximum coverage.

In [ ]:
%pip install requests pandas matplotlib --quiet

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import datetime
import urllib.parse
import json

# NOAA requires a User-Agent with contact info
HEADERS = {
    "User-Agent":  "EQMonitor/1.0 (contact@example.com)",
    "Accept":      "application/geo+json",
}

LAT, LON = -17.7134, 178.0650
MAG, DEPTH = 5.2, 226.8

TSUNAMI_MIN_MAG   = 6.0
TSUNAMI_MAX_DEPTH = 150.0
PACIFIC_RIM = {"FIJI","TONGA","VANUATU","SOLOMON","SAMOA","PACIFIC",
               "JAPAN","CHILE","INDONESIA","ALASKA","NEW ZEALAND","PHILIPPINES"}

print("✅ Config ready")

## Pre-Check Gate

Same threshold logic as NB7 (GDACS). Both sources are always polled together.

In [ ]:
region  = "FIJI REGION"
reasons = []
if MAG   < TSUNAMI_MIN_MAG:   reasons.append(f"M{MAG} < M{TSUNAMI_MIN_MAG}")
if DEPTH > TSUNAMI_MAX_DEPTH: reasons.append(f"Depth {DEPTH}km > {TSUNAMI_MAX_DEPTH}km")
if not any(r in region.upper() for r in PACIFIC_RIM): reasons.append("Not coastal")

QUERY_OK = len(reasons) == 0
print(f"Pre-check: {'✅ QUERY' if QUERY_OK else '⏭️  SKIP'}")
for r in reasons: print(f"  → {r}")
if not QUERY_OK:
    print("\n⚠️  Pre-check failed — NOAA would be skipped. Continuing to demonstrate API.")

## Fetch Active Tsunami Alerts

In [ ]:
params = urllib.parse.urlencode([
    ("event", "Tsunami Warning"),
    ("event", "Tsunami Watch"),
    ("event", "Tsunami Advisory"),
    ("status", "actual"),
])
URL = f"https://api.weather.gov/alerts/active?{params}"
print(f"Querying NOAA NWS: {URL}")

r = requests.get(URL, timeout=10, headers=HEADERS)
r.raise_for_status()
data = r.json()
print(f"✅ Response: HTTP {r.status_code} | Alerts: {len(data.get('features', []))}")

## Parse Response

In [ ]:
rows = []
for feat in data.get("features", []):
    p = feat.get("properties", {})
    rows.append({
        "event":        p.get("event"),
        "severity":     p.get("severity"),
        "urgency":      p.get("urgency"),
        "certainty":    p.get("certainty"),
        "headline":     p.get("headline"),
        "area":         p.get("areaDesc"),
        "sent":         p.get("sent"),
        "expires":      p.get("expires"),
        "status":       p.get("status"),
        "instruction":  (p.get("instruction", "") or "")[:200],
    })

df = pd.DataFrame(rows) if rows else pd.DataFrame(
    columns=["event","severity","urgency","certainty","headline","area","sent","expires","status","instruction"]
)

print(f"Active tsunami alerts: {len(df)}")
if not df.empty:
    display(df[["event","severity","urgency","area","sent","expires"]])
else:
    print("\n✅ No active NOAA tsunami alerts at this time.")

## Verdict + Combined with GDACS

In the full pipeline, NOAA and GDACS verdicts are merged. The highest severity from either source wins.

In [ ]:
SEVERITY_RANK = {"Extreme":4,"Severe":3,"Moderate":2,"Minor":1}

active = df[df['status'].str.lower() == 'actual'] if not df.empty else pd.DataFrame()

if not active.empty:
    top_idx = active['severity'].map(lambda x: SEVERITY_RANK.get(x,0)).idxmax()
    top     = active.loc[top_idx]
    verdict = f"⚠️  NOAA ALERT: [{top['severity']}] {top['event']} — {top['area']}"
    action  = "🚨 Follow official evacuation orders immediately!"
    print(verdict)
    print(f"Action:  {action}")
    if top.get('instruction'):
        print(f"\nInstruction: {top['instruction'][:200]}")
else:
    is_pac = any(r in region.upper() for r in PACIFIC_RIM)
    rl = ("HIGH"     if MAG>=7.5 and DEPTH<70  and is_pac else
          "MODERATE" if MAG>=6.5 and DEPTH<100 and is_pac else
          "LOW"      if MAG>=5.5 and DEPTH<50  and is_pac else "VERY LOW")
    print(f"✅ No active NOAA alerts — rule-based risk: {rl}")
    print(f"Action: {'Monitor GDACS/NOAA' if rl in ('MODERATE','HIGH') else 'No action needed'}")

## Visualise

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("NOAA NWS Tsunami Alerts", fontsize=12, fontweight='bold')

SEV_COLORS = {'Extreme':'#e74c3c','Severe':'#e67e22','Moderate':'#f1c40f','Minor':'#2ecc71'}
EVENT_COLORS = {'Tsunami Warning':'#e74c3c','Tsunami Watch':'#e67e22','Tsunami Advisory':'#f1c40f'}

ax = axes[0]
if not df.empty and 'event' in df.columns:
    ev_counts = df['event'].value_counts()
    colors = [EVENT_COLORS.get(e,'#aaa') for e in ev_counts.index]
    ax.barh(ev_counts.index, ev_counts.values, color=colors, edgecolor='white')
    ax.set_xlabel("Count")
    ax.set_title("Active alerts by type", fontweight='bold')
else:
    ax.text(0.5, 0.5, "No active alerts", ha='center', va='center',
            transform=ax.transAxes, fontsize=14, color='#2ecc71', fontweight='bold')
    ax.set_title("NOAA NWS Status", fontweight='bold')

# Legend patches
patches = [mpatches.Patch(color=c, label=l) for l,c in EVENT_COLORS.items()]
ax.legend(handles=patches, fontsize=8, loc='lower right')

ax = axes[1]; ax.axis('off')
status_text = "NO ACTIVE\nTSUNAMI ALERTS" if df.empty else f"{len(df)} ACTIVE ALERT(S)"
status_col  = '#2ecc71' if df.empty else '#e74c3c'
ax.text(0.5,0.80,status_text,ha='center',fontsize=16,fontweight='bold',
        color=status_col,transform=ax.transAxes)
info=[("Source","NOAA NWS api.weather.gov"),("Coverage","USA + Pacific territories"),
      ("Query interval","Every 15 min"),("Pre-check gate",str(QUERY_OK)),
      ("Alert types","Warning / Watch / Advisory")]
for i,(l,v) in enumerate(info):
    y=0.60-i*0.12
    ax.text(0.02,y,f"{l}:",fontweight='bold',fontsize=9,transform=ax.transAxes)
    ax.text(0.52,y,v,fontsize=9,transform=ax.transAxes)
ax.set_title("Query summary", fontweight='bold')

plt.tight_layout()
plt.savefig("nb8_noaa_result.png", dpi=150, bbox_inches='tight')
plt.show()
print("📊 Saved as nb8_noaa_result.png")